# core

> User facing API for solvedrive

In [ ]:
#| default_exp core

## Imports

In [ ]:
from fastcore.all import *
from fastgws import Drive as FGWSDrive
from fastgws.auth import *

import mimetypes, sys

In [ ]:
from fastcore.test import *
import asyncio

Google Drive's API is a single REST API organized around one central resource type: the **file**. Everything in Drive is a file: documents, images, even folders (a folder is a file with a special `mimeType`). Files are searched with Drive's own [query syntax](https://developers.google.com/workspace/drive/api/guides/search-files) (`name contains 'report'`, `mimeType='application/pdf'`, `'<folder-id>' in parents`), and the server returns **partial resources**. Unless you ask for specific `fields`, you get only `id`, `name`, `mimeType` and `kind` back.

A file's *content* comes in two flavors: regular binary blobs (downloaded as-is), and Google-native files (Docs, Sheets, Slides) which have no bytes of their own and must be *exported* to a regular format.

In [ ]:
#| export
FIELDS = 'id,name,mimeType,size,modifiedTime,parents,trashed,webViewLink'

class GDrive(GetAttr):
    _default = 'api'
    def __init__(self, creds): self.api = FGWSDrive(creds=creds)

    async def about(self):
        "User resource with `email` attribute"
        d = await self.api.about.get(fields='user')
        return AttrDict(d.user, email=d.user.emailAddress)

In [ ]:
creds = await oauth_creds(scopes=['https://www.googleapis.com/auth/drive'])
gd = GDrive(creds)
about = await gd.about()
me = about.email
me

'nc@answer.ai'

### Files

A file is Drive's central noun. `File` wraps the API's file resource, fetching our default `FIELDS` so the useful attributes (`name`, `mimeType`, `size`, `parents`, `trashed`, `webViewLink`) are always on hand. Anything else can be requested via `fields=`. `Folder` is a `File` whose `mimeType` marks it as a folder, and `_mk_file` picks the right class.

In [ ]:
#| export
FOLDER_MIME = 'application/vnd.google-apps.folder'

def hsize(n):
    "Human-readable byte size"
    for u in ' KMGT':
        if n < 1024: return f'{n:.0f}{u.strip()}B' if u==' ' else f'{n:.1f}{u}B'
        n /= 1024
    return f'{n:.1f}PB'

def _mtype(m): return m.split('.')[-1] if m.startswith('application/vnd.google-apps') else m

def _link(f):
    "File name as an anchor opening it in Drive in a new tab"
    nm = f.get('name') or ''
    return f'<a href="{url}" target="_blank">{nm}</a>' if (url:=f.get('webViewLink')) else nm

class File(AttrDict):
    def __init__(self, client, d):
        super().__init__(d)
        object.__setattr__(self, '_client', client)

    def _desc(self, nm):
        sz = f', {hsize(int(self.size))}' if self.get('size') else ''
        trash = '🗑' if self.get('trashed') else ''
        return f'{type(self).__name__}({self.id}: {trash}{nm} ({_mtype(self.get("mimeType", ""))}{sz}) {self.get("modifiedTime", "")})'

    def __repr__(self): return self._desc(self.get('name', ''))
    def _repr_markdown_(self): return self._desc(_link(self))

    async def refresh(self, fields=FIELDS):
        self.update(await self._client.files.get(file_id=self.id, fields=fields, supports_all_drives=True))
        return self

    @classmethod
    async def fetch(cls, client, id): return await cls(client, dict(id=id)).refresh()

class Folder(File): pass

def _fid(f): return f.id if isinstance(f, File) else f

def _mk_file(client, d):
    cls = Folder if d.get('mimeType')==FOLDER_MIME else File
    return cls(client, d)

### Uploading

Uploads go through `files.upload`, the twin fastgws generates for `files.create` from Drive's `mediaUpload` declaration. It runs Google's resumable protocol: one request registers the metadata and returns a session URI, a second sends the content to it. There is no size ceiling beyond Drive's own 5 TB, and a path streams from disk rather than loading into memory. A failed content transfer starts the upload over.

In [ ]:
#| export
@patch
async def upload(self:GDrive,
    path=None,   # File to upload
    name=None,   # Name in Drive (defaults to `path`'s name)
    data=None,   # Content bytes, as an alternative to `path`
    mime=None,   # Content type (guessed from `name` if not given)
    folder=None, # Destination `Folder` or folder id (default: My Drive root)
):
    "Upload a file to Drive, returning the new `File`"
    if path: name = name or Path(path).name
    mime = mime or mimetypes.guess_type(name)[0] or 'application/octet-stream'
    meta = dict(name=name)
    if folder: meta['parents'] = [_fid(folder)]
    r = await self.api.files.upload(media=path or data, media_type=mime, fields=FIELDS, supports_all_drives=True, **meta)
    return _mk_file(self, r)

In [ ]:
f = await gd.upload(data=b'hello from solvedrive!', name='solvedrive_test.txt')
test_eq(f.name, 'solvedrive_test.txt')
f

File(1YO_shGC2OUphCA9TZ9hvKVTZEpOXIHIo: <a href="https://drive.google.com/file/d/1YO_shGC2OUphCA9TZ9hvKVTZEpOXIHIo/view?usp=drivesdk" target="_blank">solvedrive_test.txt</a> (text/plain, 22B) 2026-07-28T20:47:43.155Z)

`Files` is a collection of `File`s with a table repr, and `search_files` runs a Drive query, returning results with our default `FIELDS` already populated (one request, no follow-up fetches needed). It searches every drive you can see; scoping to one drive comes in the Drives section.

In [ ]:
#| export
def _file_row(f):
    sz = hsize(int(f.size)) if f.get('size') else ''
    return f'| {f.id} | {"🗑" if f.get("trashed") else ""}{_link(f)} | {_mtype(f.get("mimeType") or "")} | {sz} | {(f.get("modifiedTime") or "")[:10]} |'

_file_hdr = '| id | name | type | size | modified |\n|-|-|-|-|-|\n'
class Files(L):
    def __init__(self, d, *rest, client=None, **kwargs):
        super().__init__(d, *rest, **kwargs)
        self._client = client

    def _repr_markdown_(self): return _file_hdr + '\n'.join(self.map(_file_row))

@patch
async def search_files(self:GDrive, q=None, max_results=20, fields=FIELDS, **kwargs):
    "Search every drive you can see: My Drive, files shared with you, and all shared drives"
    kwargs = dict(corpora='allDrives', include_items_from_all_drives=True, supports_all_drives=True) | kwargs
    r = await self.api.files.list(q=q, page_size=max_results, fields=f'files({fields})', **kwargs)
    return Files(L(r.get('files') or []).map(lambda o: _mk_file(self, o)), client=self)

Search takes Drive's [query syntax](https://developers.google.com/workspace/drive/api/guides/search-files) directly: `name contains 'report'`, `fullText contains 'quarterly'`, `mimeType='application/pdf'`, `modifiedTime > '2026-01-01'`, `starred=true`, combined with `and`/`or`/`not`. Trashed files match too unless you exclude them with `trashed=false`.

In [ ]:
fs = await gd.search_files("name = 'solvedrive_test.txt' and trashed=false")
fs

| id | name | type | size | modified |
|-|-|-|-|-|
| 1YO_shGC2OUphCA9TZ9hvKVTZEpOXIHIo | <a href="https://drive.google.com/file/d/1YO_shGC2OUphCA9TZ9hvKVTZEpOXIHIo/view?usp=drivesdk" target="_blank">solvedrive_test.txt</a> | text/plain | 22B | 2026-07-28 |

### Content

`download` returns a file's bytes, dispatching on the file's type so you don't have to: regular files download as-is, while Google-native files (which have no bytes of their own) are *exported*. A Doc becomes a docx, a Sheet an xlsx, a Slides deck a pptx. Pass `mime` to export to a different format (Google's [export table](https://developers.google.com/workspace/drive/api/guides/ref-export-formats) lists what's supported); it's an error on a regular file, which isn't converted. Note exports are capped at 10MB by the API.

In [ ]:
_exports = {'application/vnd.google-apps.document': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
    'application/vnd.google-apps.spreadsheet': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
    'application/vnd.google-apps.presentation': 'application/vnd.openxmlformats-officedocument.presentationml.presentation',
    'application/vnd.google-apps.drawing': 'image/png'}

@patch
async def download(self:File,
    save=None, # Optional path to also write the bytes to (a directory uses the file's name)
    mime=None, # Export format for Google-native files
):
    "File content as bytes; Google-native files are exported (docx/xlsx/pptx by default)"
    native = self.mimeType.startswith('application/vnd.google-apps')
    if not native:
        if mime: raise ValueError(f'`mime` is for exporting Google-native files; {self.name} is {self.mimeType}')
        r = await self._client.files.get(file_id=self.id, alt='media', raw_=True, supports_all_drives=True)
    else: r = await self._client.files.export(file_id=self.id, mime_type=mime or _exports[self.mimeType], raw_=True)
    data = r.content
    if save:
        p = Path(save)
        if p.is_dir(): p = p/self.name
        p.write_bytes(data)
    return data

In [ ]:
test_eq(await f.download(), b'hello from solvedrive!')

Google-native files can be created through `files.create` with the native `mimeType` (no upload needed, since there are no bytes). Exporting one gives a regular file in the requested format: docx and friends are zip containers, so the bytes start with `PK`.

In [ ]:
doc = _mk_file(gd, await gd.files.create(name='solvedrive test doc',
    mime_type='application/vnd.google-apps.document', fields=FIELDS))
docx = await doc.download()
test_eq(docx[:2], b'PK')
doc

File(1Zw7t7gHN1qW2zOBHYo-lUjB2Ezmj5FbzkPlaZVSPX7U: <a href="https://docs.google.com/document/d/1Zw7t7gHN1qW2zOBHYo-lUjB2Ezmj5FbzkPlaZVSPX7U/edit?usp=drivesdk" target="_blank">solvedrive test doc</a> (document, 1B) 2026-07-28T20:47:59.433Z)

### Folders

A folder is a file with the folder `mimeType`. It has no content, and "containment" is the child listing the folder in its `parents`. So `Folder` needs no new state: `ls` is a search scoped to `'<id>' in parents`, and `upload` drops files straight into it.

In [ ]:
@patch
async def create_folder(self:GDrive, name, parent=None):
    "Create a folder, returning the new `Folder`"
    meta = dict(name=name, mime_type=FOLDER_MIME, fields=FIELDS, supports_all_drives=True)
    if parent: meta['parents'] = [_fid(parent)]
    return Folder(self, await self.api.files.create(**meta))

@patch
async def ls(self:Folder, q=None, **kwargs):
    "Search within this folder"
    qs = f"'{self.id}' in parents and trashed=false"
    if q: qs = f'{qs} and ({q})'
    return await self._client.search_files(qs, **kwargs)

@patch
@delegates(GDrive.upload)
async def upload(self:Folder, path=None, **kwargs):
    "Upload into this folder"
    return await self._client.upload(path, folder=self, **kwargs)

In [ ]:
folder = await gd.create_folder('solvedrive tests')
f2 = await folder.upload(data=b'nested file', name='nested.txt')
kids = await folder.ls()
test_eq(kids.attrgot('id'), [f2.id])
kids

| id | name | type | size | modified |
|-|-|-|-|-|
| 1xCTlhKINFqfg1K_t-ihwBuLDoIS346cC | <a href="https://drive.google.com/file/d/1xCTlhKINFqfg1K_t-ihwBuLDoIS346cC/view?usp=drivesdk" target="_blank">nested.txt</a> | text/plain | 11B | 2026-07-28 |

### Drives

Files live in a **drive**: your personal My Drive, or a shared drive. A shared drive belongs to an organization rather than a person. Members hold one role each on the whole drive (`organizer`, `fileOrganizer`, `writer`, `commenter`, or `reader`), files have no owner, and the drive survives any member leaving. `Drive` wraps both kinds. My Drive is a synthetic entry, since the API models only shared drives as `drives` resources and addresses My Drive by the alias `root`. Every file method sends `supportsAllDrives`, so a `File` works the same wherever it lives.

In [ ]:
#| export
DRIVE_FIELDS = 'id,name,capabilities'

def _drole(d):
    "Role implied by drive `capabilities`"
    c = d.get('capabilities') or {}
    for cap,role in [('canManageMembers','organizer'),('canTrashChildren','fileOrganizer'),
        ('canAddChildren','writer'),('canComment','commenter')]:
        if c.get(cap): return role
    return 'reader'

def _dlink(d):
    "Drive name as an anchor opening it in Drive in a new tab"
    url = 'https://drive.google.com/drive/my-drive' if d.id=='root' else f'https://drive.google.com/drive/folders/{d.id}'
    return f'<a href="{url}" target="_blank">{d.name}</a>'

class Drive(AttrDict):
    def __init__(self, client, d):
        super().__init__(d)
        object.__setattr__(self, '_client', client)

    @property
    def role(self): return 'owner' if self.id=='root' else _drole(self)
    def _desc(self, nm): return f'{type(self).__name__}({self.id}: {nm}, {self.role})'
    def __repr__(self): return self._desc(self.name)
    def _repr_markdown_(self): return self._desc(_dlink(self))

def _drive_row(d): return f'| {d.id} | {_dlink(d)} | {d.role} |'
_drive_hdr = '| id | name | role |\n|-|-|-|\n'
class Drives(L):
    def __init__(self, d, *rest, client=None, **kwargs):
        super().__init__(d, *rest, **kwargs)
        self._client = client

    def _repr_markdown_(self): return _drive_hdr + '\n'.join(self.map(_drive_row))

@patch
async def list_drives(self:GDrive, fields=DRIVE_FIELDS, **kwargs):
    "Your drives: My Drive first, then every shared drive you're a member of"
    r = await self.api.drives.list(fields=f'drives({fields})', **kwargs)
    ds = [dict(id='root', name='My Drive')] + list(r.get('drives') or [])
    return Drives(L(ds).map(lambda o: Drive(self, o)), client=self)

`list_drives` returns a `Drives` table, with each name linking to the drive. This notebook creates a shared drive to work with, and filters the listing to its own artifacts. Drive changes propagate with a small delay, so these examples poll until fresh items appear:

In [ ]:
td = Drive(gd, await gd.drives.create(request_id=uuid4().hex, name='solvedrive test drive', fields=DRIVE_FIELDS))
while td.id not in (ds := await gd.list_drives()).attrgot('id'): await asyncio.sleep(1)
ds.filter(lambda d: d.id in ('root', td.id))

| id | name | role |
|-|-|-|
| root | <a href="https://drive.google.com/drive/my-drive" target="_blank">My Drive</a> | owner |
| 0ABboJZVW3csyUk9PVA | <a href="https://drive.google.com/drive/folders/0ABboJZVW3csyUk9PVA" target="_blank">solvedrive test drive</a> | organizer |

In [ ]:
#| export
@patch
async def search_files(self:Drive, q=None, **kwargs):
    "Search within this drive"
    scope = dict(corpora='user') if self.id=='root' else dict(corpora='drive', drive_id=self.id)
    return await self._client.search_files(q, **scope, **kwargs)

@patch(as_prop=True)
async def root(self:Drive):
    "The drive's root `Folder`"
    return await Folder.fetch(self._client, self.id)

A drive's `root` is an ordinary `Folder` (the API names every shared drive's root folder "Drive"), so uploading into and listing a shared drive works like any folder. `search_files` on a `Drive` scopes to that drive. On My Drive it searches the `user` corpus, which also includes files shared directly with you, since the API offers no My-Drive-only scope.

In [ ]:
tr = await td.root
f3 = await tr.upload(data=b'shared drive file', name='solvedrive_shared.txt')
while not (fs3 := await td.search_files('trashed=false')): await asyncio.sleep(1)
test_eq(fs3.attrgot('id'), [f3.id])
assert f3.id in (await gd.search_files("name = 'solvedrive_shared.txt'")).attrgot('id')
tr

Folder(0ABboJZVW3csyUk9PVA: <a href="https://drive.google.com/drive/folders/0ABboJZVW3csyUk9PVA" target="_blank">Drive</a> (folder) 2026-07-28T20:49:31.965Z)

### Organizing

Renaming is a metadata update; moving updates `parents` (`add_parents`/`remove_parents` in one call); copying is server-side, so no bytes travel through your machine. Each returns the updated (or new) `File`.

In [ ]:
@patch
async def rename(self:File, name):
    "Rename to `name`"
    return self.update(await self._client.files.update(file_id=self.id, name=name, fields=FIELDS, supports_all_drives=True))

@patch
async def move(self:File, folder):
    "Move into `folder` (a `Folder` or folder id)"
    return self.update(await self._client.files.update(file_id=self.id, add_parents=_fid(folder),
        remove_parents=','.join(self.get('parents') or []), fields=FIELDS, supports_all_drives=True))

@patch
async def copy(self:File, name=None, folder=None):
    "Server-side copy, optionally renamed into `folder`; returns the new `File`"
    meta = dict(file_id=self.id, fields=FIELDS, supports_all_drives=True)
    if name: meta['name'] = name
    if folder: meta['parents'] = [_fid(folder)]
    return _mk_file(self._client, await self._client.files.copy(**meta))

In [ ]:
await f.rename('solvedrive_renamed.txt')
await f.move(folder)
test_eq(f.name, 'solvedrive_renamed.txt')
test_eq(f.parents, [folder.id])

In [ ]:
c = await f.copy(name='solvedrive_copy.txt', folder=folder)
test_eq(await c.download(), await f.download())
c

File(10QoXi2JBENQzwBQblYQ1lYQRFxKS2jVl: <a href="https://drive.google.com/file/d/10QoXi2JBENQzwBQblYQ1lYQRFxKS2jVl/view?usp=drivesdk" target="_blank">solvedrive_copy.txt</a> (text/plain, 22B) 2026-07-28T20:50:46.132Z)

### Trash and deleting

Drive treats trashing and deletion differently. Trashing flips the `trashed` flag: the file stays (and still matches searches unless you filter `trashed=false`) until Drive purges it after 30 days, and `untrash` restores it. `delete` removes the file permanently and immediately, with no trash and no undo. Deleting a folder deletes everything inside it.

In [ ]:
@patch
async def _set_trashed(self:File, trashed):
    return self.update(await self._client.files.update(file_id=self.id, trashed=trashed, fields=FIELDS, supports_all_drives=True))

@patch
async def trash(self:File):
    "Move to the trash (recoverable for 30 days)"
    sys.audit('solvedrive.File.trash')
    return await self._set_trashed(True)

@patch
async def untrash(self:File):
    "Restore from the trash"
    return await self._set_trashed(False)

@patch
async def delete(self:File):
    "Delete permanently, skipping the trash"
    sys.audit('solvedrive.File.delete')
    return await self._client.files.delete(file_id=self.id, supports_all_drives=True)

In [ ]:
await c.trash()
assert c.trashed
await c.untrash()
assert not c.trashed
await c.delete()
test_eq(await folder.ls('name = "solvedrive_copy.txt"'), [])

### Batch operations

Operating on many files one request at a time is slow. Google's batch protocol packs up to 100 requests into one HTTP round trip, and `Files` uses it for `refresh`, `trash`, `untrash`, and `delete`, so a search result can be acted on as a unit.

In [ ]:
@patch
async def refresh(self:Files, fields=FIELDS):
    "Re-fetch metadata for all files"
    await parallel_async(File.refresh, self, fields)
    return self

@patch
async def _set_trashed(self:Files, trashed):
    await parallel_async(File._set_trashed, self, trashed)
    return self

@patch
async def trash(self:Files):
    "Move all files to the trash"
    sys.audit('solvedrive.Files.trash')
    return await self._set_trashed(True)

@patch
async def untrash(self:Files):
    "Restore all files from the trash"
    return await self._set_trashed(False)

@patch
async def delete(self:Files):
    "Delete all files permanently"
    sys.audit('solvedrive.Files.delete')
    await parallel_async(File.delete, self)

In [ ]:
ups = await asyncio.gather(*[folder.upload(data=f'test {i}'.encode(), name=f'batch_{i}.txt') for i in range(3)])
fs = Files(ups, client=gd)
await fs.trash()
test_eq(fs.attrgot('trashed'), [True]*3)
await fs.untrash()
test_eq(fs.attrgot('trashed'), [False]*3)
await fs.delete()
test_eq(await folder.ls("name contains 'batch_'"), [])

### Sharing

Access is a list of **permissions** on a file: each grants a `role` (`reader`, `commenter`, `writer`, or `owner`) to a grantee `type` (`user`, `group`, `domain`, or `anyone`). `share` grants to an email, or to anyone-with-the-link when no email is given, after which `webViewLink` is shareable.

In [ ]:
@patch
async def share(self:File,
    email=None,      # User or group to share with (anyone-with-the-link if None)
    role='reader',   # 'reader', 'commenter', 'writer', or 'owner'
    notify=False,    # Send the grantee a notification email?
):
    "Grant access, returning the new permission"
    sys.audit('solvedrive.File.share')
    meta = dict(file_id=self.id, role=role, send_notification_email=notify, supports_all_drives=True)
    if email: meta.update(type='user', email_address=email)
    else: meta['type'] = 'anyone'
    return await self._client.permissions.create(**meta)

@patch(as_prop=True)
async def permissions(self:File):
    r = await self._client.permissions.list(file_id=self.id, fields='permissions(id,type,role,emailAddress)', supports_all_drives=True)
    return r.permissions

@patch
async def unshare(self:File, permission_id):
    "Revoke a permission by id"
    return await self._client.permissions.delete(file_id=self.id, permission_id=permission_id, supports_all_drives=True)

In [ ]:
p = await f.share()
test_eq(p.type, 'anyone')
test_eq(p.role, 'reader')
await f.permissions

[{'id': 'anyoneWithLink', 'type': 'anyone', 'role': 'reader'}, {'id': '12635610242443046038', 'type': 'user', 'emailAddress': 'nc@answer.ai', 'role': 'owner'}]

In [ ]:
await f.unshare(p.id)
test_eq(len(await f.permissions), 1)

Finally, clean up everything this notebook created (deleting a folder deletes its contents; a shared drive must be emptied before deletion):

In [ ]:
await folder.delete()
await doc.delete()
await (await td.search_files()).delete()
await gd.drives.delete(drive_id=td.id)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()